In [5]:
import altair as alt
import pandas as pd
from scipy import stats
from ecostyles import EcoStyles

styles = EcoStyles()
styles.register_and_enable_theme()

GREY = "#676A86"  # ecostyles axis grey, used for the trend line and annotation

# --- Data (read off the screenshot; swap in the real series) -----------------
# Highlighted economies: name, net debt/GDP, 10y yield, label x, label y, label align
highlighted = pd.DataFrame(
    [
        ("Mexico",       45, 9.6,  48.5, 9.6,  "left"),
        ("Australia",    19, 5.4,  23.0, 5.6,  "left"),
        ("UK",           94, 5.4,  98.0, 5.6,  "left"),
        ("USA",         123, 5.2, 127.0, 5.35, "left"),
        ("Germany",      64, 3.6,  68.0, 3.2,  "left"),
        ("Japan",       249, 3.1, 245.0, 3.5,  "right"),
        ("Switzerland",  15, 0.6,  19.0, 0.45, "left"),
    ],
    columns=["economy", "net_debt", "bond_yield", "label_x", "label_y", "align"],
)
highlighted["group"] = "Highlighted"

others = pd.DataFrame(
    [(79, 9.1), (60, 7.7), (56, 7.1), (82, 7.2), (60, 6.5), (42, 5.1), (49, 4.5),
     (28, 3.5), (35, 3.3), (90, 4.0), (101, 4.1), (114, 3.9), (116, 4.8),
     (137, 4.6), (146, 4.5)],
    columns=["net_debt", "bond_yield"],
)
others["economy"] = [f"Economy {i}" for i in range(1, len(others) + 1)]
others["group"] = "Other economies"

# others first so highlighted points draw on top
df = pd.concat([others, highlighted], ignore_index=True)

# --- OLS fit -----------------------------------------------------------------
fit = stats.linregress(df["net_debt"], df["bond_yield"])
trend_df = pd.DataFrame({"net_debt": [0, 260]})
trend_df["bond_yield"] = fit.intercept + fit.slope * trend_df["net_debt"]

ann = pd.DataFrame(
    {
        "x": 262,
        "y": [9.7, 9.05, 8.65, 8.0, 7.6],
        "t": [
            f"R² = {fit.rvalue**2:.2f}",
            f"Slope {fit.slope:.3f}",
            f"p = {fit.pvalue:.2f}, n = {len(df)}",
            "No significant relationship",
            "between debt and yields",
        ],
    }
)

# --- Shared encodings (identical on every layer so axes/scales merge cleanly) -
xs = alt.Scale(domain=[0, 265], nice=False)
ys = alt.Scale(domain=[0, 10])

def xenc(field):
    return alt.X(field, type="quantitative", scale=xs, title="Net debt to GDP (%)",
                 axis=alt.Axis(values=[0, 50, 100, 150, 200, 250]))

def yenc(field):
    return alt.Y(field, type="quantitative", scale=ys, title="10-year bond yield (%)",
                 axis=alt.Axis(values=[0, 2, 4, 6, 8, 10]))

colour = alt.Color(
    "group:N", legend=None,
    scale=alt.Scale(domain=["Other economies", "Highlighted"]),  # theme palette: teal, then red
)

# --- Layers ------------------------------------------------------------------
trend = (
    alt.Chart(trend_df)
    .mark_line(strokeDash=[6, 4], strokeWidth=1.8, color=GREY)
    .encode(x=xenc("net_debt"), y=yenc("bond_yield"))
)

points = (
    alt.Chart(df)
    .mark_circle(size=90, opacity=1)
    .encode(
        x=xenc("net_debt"), y=yenc("bond_yield"), color=colour,
        tooltip=["economy:N", "net_debt:Q", "bond_yield:Q"],
    )
)

def labels(align):
    return (
        alt.Chart(highlighted[highlighted["align"] == align])
        .mark_text(align=align, baseline="middle", fontSize=12, fontWeight="bold")
        .encode(x=xenc("label_x"), y=yenc("label_y"), text="economy:N", color=colour)
    )

def note(rows, **mark):
    return (
        alt.Chart(rows)
        .mark_text(align="right", baseline="middle", color=GREY, **mark)
        .encode(x=xenc("x"), y=yenc("y"), text="t:N")
    )

chart = (
    trend + points + labels("left") + labels("right")
    + note(ann.iloc[:1], fontSize=20, fontWeight="bold")
    + note(ann.iloc[1:], fontSize=12)
).properties(
    width=400,
    height=300,
    
)

styles.save(chart, path=".", name="debt_and_bond_yields", svg=True)
chart

alt.LayerChart(...)